Precision: set `LATENT_RELAY_INFER_DTYPE=fp16` **before** the first model load. Alignment and OBF stay in FP32.

In [ ]:
import os
import json
import shutil
import sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [
    Path.cwd(),
    Path("/kaggle/input/latent-relay-audit"),
    Path("/kaggle/input/latentrelayaudit"),
]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_a.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)
REPO = None
for cand in candidates:
    if (cand / "src" / "stage_a.py").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError(
        "Could not find src/stage_a.py under /kaggle/input. "
        "Check that the dataset contains the repo (src/stage_a.py)."
    )

if ON_KAGGLE and REPO != WORK / "latent-relay-audit":
    dest = WORK / "latent-relay-audit"
    if not (dest / "src" / "stage_a.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print("REPO", REPO.resolve())
print("ON_KAGGLE", ON_KAGGLE)
print("LATENT_RELAY_INFER_DTYPE", os.environ["LATENT_RELAY_INFER_DTYPE"])

In [ ]:
import subprocess
import sys

# Pin the KV-cache interface. Keep Kaggle's torch/CUDA wheel.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
print("transformers pin installed")

In [ ]:
import torch, transformers
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0), "cap", torch.cuda.get_device_capability(0))
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57"), transformers.__version__

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])

## Persistence check

Write a tiny file now. After the session stops, download it from the notebook output. If that fails, later JSONL results will not survive either.

In [ ]:
persist = WORK / "stage_a_persistence_check.json"
persist.write_text(json.dumps({"ok": True, "repo": str(REPO)}, indent=2), encoding="utf-8")
print("wrote", persist)
print("Download this file after the session ends before trusting Stage C JSONL survival.")

## Run Stage A

Order: FP16 policy (already set) → alignment-matrix probe → `full` → `headwise` → `hobf_fast`.
Do not conclude that Qwen3-4B fits until alignment init and OBF have both run.

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_a"])
print("stage_a exit", rc)

In [ ]:
from pathlib import Path
stage_dirs = sorted((REPO / "results" / "stage_a").glob("*"))
stage_dirs = [p for p in stage_dirs if p.is_dir()]
print("stage_a dirs", stage_dirs)
if stage_dirs:
    latest = stage_dirs[-1]
    summary = latest / "summary.json"
    print(summary.read_text(encoding="utf-8")[:4000])
    if ON_KAGGLE:
        dest = WORK / "results_stage_a"
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(latest, dest)
        print("copied results to", dest)